# World Cup Predictor Model 2026
By Karl Estampador :)

## Model description

This notebook loads the 2026 Elo ratings and exposes three functions used by `predictions.ipynb`:

| Function | Returns |
|---|---|
| `get_elo(team_name)` | Elo rating for a team |
| `predict_winner(home, away)` | `(winner, loser, winner_elo, loser_elo)` — higher Elo always wins |
| `predict_score(home, away)` | `(home_goals, away_goals)` — winner scores 2; loser 0 (or 1-0 if Elo diff < 50) |

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path('data')

## 1. Load Elo ratings (based on Elo ratings from May 27, 2026)

In [2]:
elo_raw = pd.read_csv(DATA / 'elo_ratings_wc2026.csv')
elo_2026 = (
    elo_raw[elo_raw['snapshot_date'] == '2026-05-27']
    .copy()
    .reset_index(drop=True)
)
print(f'{len(elo_2026)} teams in 2026-05-27 Elo snapshot')
elo_2026[['country', 'rating', 'rank']].sort_values('rank').head(10)

48 teams in 2026-05-27 Elo snapshot


,country,rating,rank
0,Spain,2165,1
1,Argentina,2113,2
2,France,2081,3
3,England,2020,4
4,Brazil,1984,5
5,Portugal,1984,5
6,Colombia,1975,7
7,Netherlands,1961,8
8,Ecuador,1933,9
9,Croatia,1930,10


## 2. Load teams and FIFA rank tiebreaker

In [3]:
teams_df = pd.read_csv(DATA / 'teams.csv')
test_df  = pd.read_csv(DATA / 'test.csv')

# fifa_rank_pre_tournament is in test.csv, keyed by team name
fifa_rank = dict(zip(test_df['team'], test_df['fifa_rank_pre_tournament']))
print(f'{len(teams_df)} teams loaded; {len(fifa_rank)} FIFA ranks available')
teams_df.head()

48 teams loaded; 48 FIFA ranks available


,id,team_name,fifa_code,group_letter,is_placeholder
0,1,Mexico,MEX,A,False
1,2,South Africa,RSA,A,False
2,3,South Korea,KOR,A,False
3,4,Czechia,CZE,A,False
4,5,Canada,CAN,B,False


## 3. Team name alias map

Three naming schemes exist across files. This map normalises any source name to the key used in `elo_2026['country']`.

In [4]:
# maps any variant name to the country name in the elo database
ALIASES: dict[str, str] = {
    # teams.csv / matches.csv -> elo country
    'USA':                      'United States',
    'IR Iran':                  'Iran',
    "Côte d'Ivoire":            'Ivory Coast',
    'Cabo Verde':               'Cape Verde',
    'DR Congo':                 'DR Congo',       
    # test.csv / train.csv variants
    'United States':            'United States',
    'Iran':                     'Iran',
    'Ivory Coast':              'Ivory Coast',
    'Cape Verde':               'Cape Verde',
    'Czech Republic':           'Czechia',
    'Cura\u00e7ao':             'Cura\u00e7ao', # special c character
}

# Build Elo lookup: canonical name -> rating
_elo_lookup: dict[str, float] = dict(zip(elo_2026['country'], elo_2026['rating']))

# Build FIFA rank lookup accepting any variant name
_rank_lookup: dict[str, float] = {}
for name, rank in fifa_rank.items():
    _rank_lookup[name] = rank
    canon = ALIASES.get(name, name)
    _rank_lookup[canon] = rank

# resolve any team name variant to the canonical Elo country name
def _canonical(name: str) -> str:
    return ALIASES.get(name, name)

# returns pre-tournament Elo rating for team name
def get_elo(team_name: str) -> float:
    canon = _canonical(team_name)
    if canon in _elo_lookup:
        return _elo_lookup[canon]
    # Fallback: try the original name
    if team_name in _elo_lookup:
        return _elo_lookup[team_name]
    raise KeyError(f'No Elo rating found for "{team_name}" (canonical: "{canon}")')

# gets FIFA rank for a tiebreaker if elo is the same
def _get_fifa_rank(team_name: str) -> float:
    canon = _canonical(team_name)
    for key in (team_name, canon):
        if key in _rank_lookup:
            v = _rank_lookup[key]
            try:
                return float(v)
            except (TypeError, ValueError):
                pass
    return 999.0 # if we can't find FIFA rank


print('Alias map and Elo lookup ready.')
# Quick sanity check to make sure elo lookup is working
for t in ['Spain', 'USA', 'IR Iran', "Côte d'Ivoire", 'Bosnia and Herzegovina', 'Iraq', 'DR Congo', 'Czechia', 'Sweden', 'Turkey']:
    print(f'  {t:35s} Elo={get_elo(t):.0f}')

Alias map and Elo lookup ready.
  Spain                               Elo=2165
  USA                                 Elo=1721
  IR Iran                             Elo=1760
  Côte d'Ivoire                       Elo=1676
  Bosnia and Herzegovina              Elo=1594
  Iraq                                Elo=1607
  DR Congo                            Elo=1655
  Czechia                             Elo=1726
  Sweden                              Elo=1719
  Turkey                              Elo=1902


## 4. Core model functions

In [5]:
# predicts the winner of a match between home and away, higher elo always wins,
# then lower FIFA rank wins. 
def predict_winner(
    home: str, away: str
) -> tuple[str, str, float, float]:

    home_elo = get_elo(home)
    away_elo = get_elo(away)

    if home_elo > away_elo:
        return home, away, home_elo, away_elo
    elif away_elo > home_elo:
        return away, home, away_elo, home_elo
    else:
        # Extremely rare exact tie: use FIFA rank
        home_rank = _get_fifa_rank(home)
        away_rank = _get_fifa_rank(away)
        if home_rank <= away_rank:
            return home, away, home_elo, away_elo
        else:
            return away, home, away_elo, home_elo

# produce a scoreline for group-stage use
def predict_score(home: str, away: str) -> tuple[int, int]:
    winner, loser, w_elo, l_elo = predict_winner(home, away)
    diff = abs(w_elo - l_elo)
    goals_w = 2 if diff >= 50 else 1
    goals_l = 0
    if winner == home:
        return goals_w, goals_l
    else:
        return goals_l, goals_w


print('predict_winner and predict_score ready.')

# Demo
demo_matches = [
    ('Spain',     'Argentina'),
    ('Argentina', 'Jordan'),
    ('USA',       'Turkey'),
    ('France',    'Iraq'),
]
print(f'\n{"Match":<35} {"Winner":<30} {"Score"}')
print('-' * 75)
for h, a in demo_matches:
    w, l, we, le = predict_winner(h, a)
    hg, ag = predict_score(h, a)
    print(f'{h} vs {a:<25} {w:<30} {hg}-{ag}  (Elo: {we:.0f} vs {le:.0f})')

predict_winner and predict_score ready.

Match                               Winner                         Score
---------------------------------------------------------------------------
Spain vs Argentina                 Spain                          2-0  (Elo: 2165 vs 2113)
Argentina vs Jordan                    Argentina                      2-0  (Elo: 2113 vs 1690)
USA vs Turkey                    Turkey                         0-2  (Elo: 1902 vs 1721)
France vs Iraq                      France                         2-0  (Elo: 2081 vs 1607)
